![Token-Gated Language Models](https://raw.githubusercontent.com/token-gate-llms/TokenGate/main/assets/title.png)

[![GitHub](https://img.shields.io/badge/GitHub-TokenGate-181717?logo=github&logoColor=white)](https://github.com/token-gate-llms/TokenGate) [![arXiv](https://img.shields.io/badge/arXiv-2610.26754-B31B1B?logo=arxiv&logoColor=white)](https://arxiv.org/abs/2610.26754) [![X](https://img.shields.io/badge/X-Launch%20thread-000000?logo=x&logoColor=white)](https://x.com/lianapatel_/status/2112924355240099875) [![HF Qwen3](https://img.shields.io/badge/%F0%9F%A4%97-Qwen3--1.7B-FFD21E)](https://huggingface.co/liana313/tokengate-niah-qwen3-1.7b) [![HF Qwen3.5](https://img.shields.io/badge/%F0%9F%A4%97-Qwen3.5--2B-FFD21E)](https://huggingface.co/liana313/tokengate-niah-qwen3.5-2b)

Current language models read all tokens of their input context, processing each one through all model layers. **TokenGate lets a model skim:** a learned gate selects the document tokens that matter for the question, so later layers process fewer tokens. It extends an existing model with lightweight adaptation; the question and instructions remain available to the model.

In this demo, we will ask a model to find facts hidden in long documents, first with ordinary inference and then with a released TokenGate checkpoint. We will compare answers and prefill time across token budgets, inspect what the gate kept, and optionally train a gate and a separate LoRA baseline ourselves. The default inference path is designed for a free Colab T4; the full training path needs A100 GPUs.

![Animation: TokenGate selects document tokens after an early layer, so later layers process fewer tokens](https://raw.githubusercontent.com/token-gate-llms/TokenGate/0a287fbbd4bfeff6df566a0eae4b1e3134f21c28/assets/tokengate_explainer.gif)

The [paper](https://arxiv.org/abs/2610.26754) reports the following matched-accuracy results for Qwen3-1.7B on A100 GPUs, against a separately LoRA-tuned model using full prefill. These are published benchmark numbers; the demo below measures your own run.

<p align="center"><img src="https://raw.githubusercontent.com/token-gate-llms/TokenGate/main/assets/results_figure.png" width="700" alt="Published TokenGate speed and memory results for NIAH and RepoBench"></p>


### The paper's Pareto frontier

Across five tasks, this annotated [paper figure](https://arxiv.org/html/2610.26754v1#S5) compares accuracy with time to first token (top) and peak prefill memory (bottom). Red is TokenGate, blue is the separately LoRA-tuned full-prefill baseline, and gray is other methods. These A100 results provide context for the smaller live demo below.

<p align="center"><img src="https://raw.githubusercontent.com/token-gate-llms/TokenGate/0e7af59e4e8cbf7b6ad2db33f1875ce9eccf45f9/assets/tokengate_results_annotated.png" width="1000" alt="Accuracy versus time to first token and peak prefill memory across five tasks"></p>


## 1. Setup and information

We will use **needle-in-a-haystack (NIAH)**: place one fact in a long document, ask a question about it, and check whether the model retrieves the fact. Our three test examples ask about different cities, with the fact near the beginning, middle, or end. At the default 16k length, they come from the frozen 150-row test partition rather than the training rows.

Choose the model and document length below. **Token budgets** are the maximum numbers of *document* tokens the gate may pass to later layers; try several to see the speed and accuracy tradeoff. Select your actual GPU runtime, then run the install cell to load the repository API and model dependencies. Qwen3-1.7B on T4 is the default. The ordinary Qwen3 T4 baseline uses a single pass through 4k and cached chunks above 4k to fit in memory; we will label that timing in the results. Qwen3.5 runs through 8k on T4, while 16k requires an A100.

The API follows a familiar model workflow: make a prompt with `apply_chat_template`, load a checkpoint with `load_model`, then call `model.generate`. The code below shows those calls. The demo helpers keep data preparation, repeated measurements, and plots out of the main cells.


In [ ]:
#@title Demo settings { display-mode: "form" }
MODEL = 'Qwen3-1.7B' #@param ['Qwen3-1.7B', 'Qwen3.5-2B']
DOC_TOKENS = '16k' #@param ['2k', '4k', '8k', '16k']
TOKEN_BUDGETS = '100, 500, 2000' #@param {type:"string"}
GPU_RUNTIME = 'T4' #@param ['T4', 'A100']

BASE_MODELS = {'Qwen3-1.7B':'Qwen/Qwen3-1.7B', 'Qwen3.5-2B':'Qwen/Qwen3.5-2B'}
CHECKPOINTS = {'Qwen3-1.7B':('liana313/tokengate-niah-qwen3-1.7b','3ef4d84ebd161a9700acced6e92f7cd5c50225ad'),
               'Qwen3.5-2B':('liana313/tokengate-niah-qwen3.5-2b','b1fe3e72b9896efc821fbfa02a19fcded4de6d77')}
BASE_MODEL = BASE_MODELS[MODEL]
DOC_TOKENS = int(DOC_TOKENS[:-1]) * 1024
assert MODEL != 'Qwen3.5-2B' or GPU_RUNTIME == 'A100' or DOC_TOKENS <= 8192, 'On a T4, Qwen3.5 supports up to 8k; choose an A100 for 16k.'
BUDGETS = tuple(int(x.strip()) for x in TOKEN_BUDGETS.split(',') if x.strip())
assert BUDGETS and len(set(BUDGETS)) == len(BUDGETS) and all(k > 0 for k in BUDGETS)
print(BASE_MODEL, '|', DOC_TOKENS, 'document tokens | K =', BUDGETS, '| GPU:', GPU_RUNTIME)

In [ ]:
#@title Install and import the API { display-mode: "form" }
import gc, importlib, logging, os, subprocess, sys, warnings
os.environ['HF_HUB_VERBOSITY'] = 'error'
logging.getLogger('huggingface_hub').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', message='You are sending unauthenticated requests to the HF Hub.*')
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Choose a GPU runtime first.'
assert GPU_RUNTIME in torch.cuda.get_device_name(0), f'Select a {GPU_RUNTIME} GPU in Colab: Runtime → Change runtime type.'
SOURCE = Path('/content/TokenGate') if Path('/content').is_dir() else Path.cwd()
if Path('/content').is_dir():
    if not SOURCE.exists():
        subprocess.run(['git','clone','-q','--branch','tokengate_demo',
                        'https://github.com/token-gate-llms/TokenGate',str(SOURCE)],check=True)
    else:
        subprocess.run(['git','-C',str(SOURCE),'fetch','origin','tokengate_demo'],check=True)
        subprocess.run(['git','-C',str(SOURCE),'switch','-C','tokengate_demo',
                        'origin/tokengate_demo'],check=True)
    local_head = subprocess.check_output(['git','-C',str(SOURCE),'rev-parse','HEAD'],text=True).strip()
    remote_head = subprocess.check_output(['git','-C',str(SOURCE),'rev-parse','origin/tokengate_demo'],text=True).strip()
    assert local_head == remote_head, 'The Colab checkout did not update to the latest demo branch.'
assert (SOURCE/'tokengate/demo.py').exists(), 'TokenGate demo helper is missing from this checkout.'
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(SOURCE),
                'transformers==5.5.0','matplotlib'],check=True)
sys.path.insert(0,str(SOURCE))
# A Colab runtime can keep the previous editable install in sys.modules.
for name in list(sys.modules):
    if name == 'tokengate' or name.startswith('tokengate.'):
        del sys.modules[name]
importlib.invalidate_caches()
from transformers import AutoTokenizer, AutoModelForCausalLM
if MODEL == 'Qwen3.5-2B':
    from transformers.models.qwen3_5.modeling_qwen3_5 import is_fast_path_available
    if not is_fast_path_available:
        logging.getLogger('transformers.models.qwen3_5.modeling_qwen3_5').setLevel(logging.ERROR)
        print('Qwen3.5: PyTorch DeltaNet fallback is active; baseline and gate timings include it.')
from huggingface_hub import snapshot_download
from tokengate import (apply_chat_template, split_state_span, load_model,
                       TokenGateConfig, TokenGateTrainer)
from tokengate.demo import (niah_examples, niah_prompt_text, niah_prompt,
                            plain_results, gate_results,
                            result_figures, show_kept_tokens, release_niah_rows,
                            release_niah_config_kwargs)
print('GPU:', torch.cuda.get_device_name(0))

## 2. Inference

We will give the *same three questions* to the ordinary model and TokenGate, then compare their answers and prefill times. Each question has one fact buried at a different depth in its document.

### Ordinary model: read the whole document

First, make the held-out prompts. The `{state}` placeholder marks the document that TokenGate may prune; `apply_chat_template(..., state=...)` inserts its text. `split_state_span` removes TokenGate's markers to give the ordinary model the same full document. The printed city and depth tell you where each answer is hidden. Then run the unmodified base model: `correct` checks its answer against the planted fact, and **TTFT** is time to first token. These times will be our baseline.


In [ ]:
# Make three held-out NIAH prompts
samples = niah_examples(SOURCE, DOC_TOKENS)
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
print("Prompt template:", niah_prompt_text(samples[0]))
gate_prompts = [apply_chat_template(tok, niah_prompt_text(row), state=row['body'])
                for row in samples]
plain_prompts = [split_state_span(ids)[0] for ids in gate_prompts]
for row in samples:
    print(f"{row['city']:7s} needle at {row['depth_actual']:.0%} · {row['question']}")


In [ ]:
#@title Run the ordinary model { display-mode: "form" }
if GPU_RUNTIME == 'A100':
    assert torch.cuda.get_device_properties(0).total_memory >= 35*2**30, 'Use an A100-class GPU for single-pass prefill.'
dtype = torch.float16 if GPU_RUNTIME == 'T4' else torch.bfloat16
CHUNK_THRESHOLD = 4096  # Qwen3 on T4: chunk only above 4k document tokens
PLAIN_CHUNK_TOKENS = 1024 if GPU_RUNTIME == 'T4' and MODEL == 'Qwen3-1.7B' and DOC_TOKENS > CHUNK_THRESHOLD else None
print('Ordinary-model prefill:', f'cached {PLAIN_CHUNK_TOKENS}-token chunks (Qwen3/T4, above 4k)' if PLAIN_CHUNK_TOKENS else 'single pass')
plain_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=dtype,
    attn_implementation='sdpa').cuda().eval()
plain = plain_results(plain_model, tok, plain_prompts, samples,
    chunk_tokens=PLAIN_CHUNK_TOKENS)
for row in plain: print(f"{row['city']:7s} correct={bool(row['correct'])}  TTFT={row['prefill_s']:.2f}s")
del plain_model; gc.collect(); torch.cuda.empty_cache()

### Accelerate with TokenGate

`load_model` combines the base model, trained gate, and paired LoRA adapter from the released checkpoint. The next cell calls `gate.generate` on one question so you can see the public API directly. `max_kept_prefill_tokens=K` is the document-token budget; smaller K means fewer document tokens continue through later layers. The helper then repeats this call across all three questions and budgets for timed results. Compare `correct`, TTFT, and the retained fraction at each K.


In [ ]:
# Load a checkpoint, try the API, then benchmark each token budget
repo_id, revision = CHECKPOINTS[MODEL]
checkpoint = snapshot_download(repo_id, revision=revision)
gate = load_model(checkpoint, gate_attn='sdpa' if GPU_RUNTIME == 'T4' else 'flex', dtype=dtype)
example_out = gate.generate(gate_prompts[0], tok, max_kept_prefill_tokens=BUDGETS[0],
                            max_new_tokens=64)
print(f"API example: K={BUDGETS[0]}, answer={example_out.answers[0]!r}")
gated = gate_results(gate, tok, gate_prompts, samples, BUDGETS)
for row in gated:
    print(f"{row['city']:7s} K={str(row['budget']):5s} correct={bool(row['correct'])} "
          f"TTFT={row['prefill_s']:.2f}s retained={row['retained']:.1%}")

### Interpret the results

The first plot compares **mean prefill time** for the ordinary model and each gate budget; lower is faster. The second divides the baseline time by each gate time: **1×** means equal speed, and **2×** means the gate is twice as fast. Check the `correct` lines above alongside either plot, since a faster answer is useful only if it still retrieves the fact.

For Qwen3 on T4, the ordinary baseline uses cached chunks **above 4k**; the paper measures single-pass prefill, so these 8k/16k T4 speedups are not directly comparable to its numbers. Qwen3.5 uses a single pass through 8k on T4, and A100 uses one at every available length. Three examples show how the method behaves; they are too few for a reliable accuracy estimate.


In [ ]:
#@title Prefill time and speedup { display-mode: "form" }
from IPython.display import display
for fig in result_figures(plain, gated,
    baseline_name='chunked base (T4)' if PLAIN_CHUNK_TOKENS else f'single-pass base ({GPU_RUNTIME})'):
    display(fig)

### See what the gate selected

After the plots, inspect the first question at the smallest budget. The token view shows which parts of its document survived pruning, so you can see whether the planted fact was kept.

In [ ]:
#@title Which document tokens did the gate keep? { display-mode: "form" }
show_kept_tokens(gate, tok, gate_prompts[0], samples[0], budget=min(BUDGETS))

## 3. Optional: train and compare your own models

The inference above uses released weights. To train your own gate, switch to an A100 runtime, enable `RUN_TRAINING`, and run the next cell. It follows the README's `TokenGateConfig` → `TokenGateTrainer` → `trainer.train()` API. The recipe helper fills in the published NIAH settings; the trainer takes rows with `article` (the prunable document) and `answer` fields. This run trains a gate on 12,000 NIAH rows, with 50 validation rows and 150 held out for testing. It also trains a **separate LoRA-only baseline** for comparison. Four A100s match the paper's batch layout; fewer GPUs take longer, and the full run can take hours.

Then run the final cell to compare your trained gate against your trained LoRA baseline on three held-out test questions. Its plots use the LoRA model's prefill time as the denominator, matching the paper's type of baseline. The three-question result is a quick check of your run, not the full paper benchmark.


In [ ]:
# Train TokenGate and the matched LoRA baseline (A100 only)
RUN_TRAINING = False #@param {type:"boolean"}
if RUN_TRAINING:
    assert GPU_RUNTIME == 'A100', 'Choose A100 in Demo settings and switch to an A100 runtime.'
    n_gpus = torch.cuda.device_count()
    assert n_gpus in (1, 2, 4) and all('A100' in torch.cuda.get_device_name(i) for i in range(n_gpus)), 'Use 1, 2, or 4 visible A100s.'
    del gate
    gc.collect(); torch.cuda.empty_cache()
    work = Path('/content/tokengate-training') if Path('/content').is_dir() else Path.cwd()/'tokengate-training'
    work = work/MODEL
    data_root = work/'data'
    train_rows, eval_rows = release_niah_rows(SOURCE, work)
    recipe = release_niah_config_kwargs(SOURCE, BASE_MODEL, work/'gate', data_root, n_gpus)
    shown = ("base_model", "gate_layer", "learning_rate", "epochs",
             "max_steps", "k_eval", "lora_rank")
    print("TokenGateConfig:", {key: recipe[key] for key in shown})
    config = TokenGateConfig(**recipe)
    trainer = TokenGateTrainer(config, train_dataset=train_rows, eval_dataset=eval_rows, task='niah_kamradt')
    gate_checkpoint = Path(config.output_dir)/'gate_final.pt'
    if not gate_checkpoint.exists():
        if MODEL == 'Qwen3-1.7B':
            from kernels.fa2bias.keybias import available as fa2_available
            if not fa2_available():
                subprocess.run([sys.executable, str(SOURCE/'kernels/fa2bias/build.py')], check=True)
        trainer.train()
    del trainer, train_rows, eval_rows
    gc.collect()
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'peft>=0.15'], check=True)
    baseline_dir = work/'lora_baseline'
    if not (baseline_dir/'adapter_model.safetensors').exists():
        baseline_steps = 10000 if MODEL == 'Qwen3-1.7B' else 18000
        subprocess.run([sys.executable, '-m', 'torch.distributed.run',
                        f'--nproc_per_node={n_gpus}', '--master_port=29501',
                        '-m', 'tokengate.demo_baseline', '--base-model', BASE_MODEL,
                        '--data-root', str(data_root), '--eval-split', str(config.eval_split),
                        '--output-dir', str(baseline_dir), '--max-steps', str(baseline_steps),
                        '--grad-accum', str(4//n_gpus)], check=True)
    print('Gate:', gate_checkpoint, '| LoRA baseline:', baseline_dir)
else:
    print('Optional full training skipped; inference above used the released checkpoint.')


In [ ]:
#@title Compare your trained gate with its LoRA baseline (A100 only) { display-mode: "form" }
if RUN_TRAINING:
    from peft import PeftModel
    benchmark_rows = niah_examples(SOURCE, 16384)
    pairs = [niah_prompt(tok, row) for row in benchmark_rows]
    trained_gate_prompts, baseline_prompts = map(list, zip(*pairs))
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.bfloat16, attn_implementation='sdpa')
    tuned_base = PeftModel.from_pretrained(base, str(baseline_dir)).merge_and_unload().cuda().eval()
    tuned_plain = plain_results(tuned_base, tok, baseline_prompts, benchmark_rows)
    print('LoRA baseline:', sum(r['correct'] for r in tuned_plain), '/', len(tuned_plain), 'correct')
    del tuned_base, base
    gc.collect(); torch.cuda.empty_cache()
    own_gate = load_model(str(gate_checkpoint), gate_attn='fa2' if MODEL == 'Qwen3-1.7B' else 'flex', dtype=torch.bfloat16)
    own = gate_results(own_gate, tok, trained_gate_prompts, benchmark_rows, BUDGETS)
    for k in BUDGETS:
        rows = [r for r in own if r['budget'] == k]
        print(f'K={k}: {sum(r["correct"] for r in rows)}/{len(rows)} correct')
    for fig in result_figures(tuned_plain, own, baseline_name='LoRA-tuned base (SDPA, A100)'):
        display(fig)
else:
    print('Run the optional training cell first.')
